# Data Processing: Parsing + Validation

- Parsing, Validation and Feature calculation was processed inside a local terminal to prevent crashing the ipykernel due to RAM usage
- The event parquet has over 124 million rows and 75 columns. 
- The output of the validation script is shown. 
- Samples of the events parquet are used to identify the shape, data and columns. 

 src % python3 validation.py
=== Validierungsreport (pro Handelstag, über alle Row Groups aggregiert) ===
trading_day     rows  monotonic_violations  crossed_book  top_of_book_nans  bad_action_values  bad_symbol_values
   20260601  5894337                     0             0                18                  0                  0
   20260602  5292053                     0             0                 3                  0                  0
   20260603  5390806                     0             0                 1                  0                  0
   20260604  5455608                     0             0                12                  0                  0
   20260605  8274654                     0             0                 8                  0                  0
   20260608  5195880                     0             0                12                  0                  0
   20260609 10490258                     0             0                 2                  0                  0
   20260610  8727910                     0             0                 2                  0                  0
   20260611  6721346                     0             0                 1                  0                  0
   20260612  5556117                     0             0                 4                  0                  0
   20260615  4199649                     0             0                39                  0                  0
   20260616  4678107                     0             0                12                  0                  0
   20260617  5263985                     0             0                 6                  0                  0
   20260618  4738646                     0             0                 7                  0                  0
   20260622  4527300                     0             0                79                  0                  0
   20260623  5453318                     0             0                 7                  0                  0
   20260624  6380941                     0             0                 3                  0                  0
   20260625  7065195                     0             0                10                  0                  0
   20260626  5141809                     0             0                 9                  0                  0
   20260629  5519045                     0             0                29                  0                  0
   20260630  4132620                     0             0                10                  0                  0

Zeilen gesamt:          124,099,584
Summe Crossed Book:     0
Summe Monoton-Verstöße: 0
Summe Top-of-Book NaNs: 274

In [2]:
import polars as pl
import pandas as pd

df = (
    pl.scan_parquet("../data/processed/nvda_202606_events.parquet")
    .head(1000)
    .collect()
)

print(df)

shape: (1_000, 75)
┌─────────────┬─────────────┬───────┬────────────┬───┬───────────┬───────────┬────────┬────────────┐
│ ts_recv     ┆ ts_event    ┆ rtype ┆ publisher_ ┆ … ┆ bid_ct_09 ┆ ask_ct_09 ┆ symbol ┆ trading_da │
│ ---         ┆ ---         ┆ ---   ┆ id         ┆   ┆ ---       ┆ ---       ┆ ---    ┆ y          │
│ datetime[ns ┆ datetime[ns ┆ i64   ┆ ---        ┆   ┆ f64       ┆ f64       ┆ str    ┆ ---        │
│ , UTC]      ┆ , UTC]      ┆       ┆ i64        ┆   ┆           ┆           ┆        ┆ str        │
╞═════════════╪═════════════╪═══════╪════════════╪═══╪═══════════╪═══════════╪════════╪════════════╡
│ 2026-06-01  ┆ 2026-06-01  ┆ 10    ┆ 2          ┆ … ┆ 0.0       ┆ 0.0       ┆ NVDA   ┆ 20260601   │
│ 08:00:00.02 ┆ 08:00:00.02 ┆       ┆            ┆   ┆           ┆           ┆        ┆            │
│ 3759450 …   ┆ 3591265 …   ┆       ┆            ┆   ┆           ┆           ┆        ┆            │
│ 2026-06-01  ┆ 2026-06-01  ┆ 10    ┆ 2          ┆ … ┆ 0.0       ┆ 0.0  

In [3]:
df.shape
df.head()

ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,flags,ts_in_delta,sequence,bid_px_00,ask_px_00,bid_sz_00,ask_sz_00,bid_ct_00,ask_ct_00,bid_px_01,ask_px_01,bid_sz_01,ask_sz_01,bid_ct_01,ask_ct_01,bid_px_02,ask_px_02,bid_sz_02,ask_sz_02,bid_ct_02,ask_ct_02,bid_px_03,ask_px_03,bid_sz_03,ask_sz_03,bid_ct_03,ask_ct_03,bid_px_04,ask_px_04,bid_sz_04,ask_sz_04,bid_ct_04,ask_ct_04,bid_px_05,ask_px_05,bid_sz_05,ask_sz_05,bid_ct_05,ask_ct_05,bid_px_06,ask_px_06,bid_sz_06,ask_sz_06,bid_ct_06,ask_ct_06,bid_px_07,ask_px_07,bid_sz_07,ask_sz_07,bid_ct_07,ask_ct_07,bid_px_08,ask_px_08,bid_sz_08,ask_sz_08,bid_ct_08,ask_ct_08,bid_px_09,ask_px_09,bid_sz_09,ask_sz_09,bid_ct_09,ask_ct_09,symbol,trading_day
"datetime[ns, UTC]","datetime[ns, UTC]",i64,i64,i64,str,str,i64,f64,f64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str
2026-06-01 08:00:00.023759450 UTC,2026-06-01 08:00:00.023591265 UTC,10,2,11667,"""A""","""N""",0,215.77,100.0,128,168185,309052,215.77,null,100.0,0.0,1.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,"""NVDA""","""20260601"""
2026-06-01 08:00:00.025605519 UTC,2026-06-01 08:00:00.025436081 UTC,10,2,11667,"""T""","""A""",0,215.77,100.0,0,169438,309230,215.77,null,100.0,0.0,1.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,"""NVDA""","""20260601"""
2026-06-01 08:00:00.025605519 UTC,2026-06-01 08:00:00.025436081 UTC,10,2,11667,"""C""","""B""",0,215.77,100.0,128,169438,309230,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,"""NVDA""","""20260601"""
2026-06-01 08:00:00.035527202 UTC,2026-06-01 08:00:00.035360050 UTC,10,2,11667,"""A""","""B""",0,211.63,2.0,128,167152,310329,211.63,null,2.0,0.0,1.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,"""NVDA""","""20260601"""
2026-06-01 08:00:00.035584724 UTC,2026-06-01 08:00:00.035417587 UTC,10,2,11667,"""A""","""B""",1,206.1,2.0,128,167137,310336,211.63,null,2.0,0.0,1.0,0.0,206.1,null,2.0,0.0,1.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,null,null,0.0,0.0,0.0,0.0,"""NVDA""","""20260601"""


# Feature Set

In [4]:
feat = pd.read_parquet("../data/processed/nvda_202606_features.parquet")


In [5]:
feat.shape

(1409879, 92)

In [9]:
for i, col in enumerate(feat.columns):
    print(i, col)

0 ts_recv
1 ts_event
2 rtype
3 publisher_id
4 instrument_id
5 action
6 side
7 depth
8 price
9 size
10 flags
11 ts_in_delta
12 sequence
13 bid_px_00
14 ask_px_00
15 bid_sz_00
16 ask_sz_00
17 bid_ct_00
18 ask_ct_00
19 bid_px_01
20 ask_px_01
21 bid_sz_01
22 ask_sz_01
23 bid_ct_01
24 ask_ct_01
25 bid_px_02
26 ask_px_02
27 bid_sz_02
28 ask_sz_02
29 bid_ct_02
30 ask_ct_02
31 bid_px_03
32 ask_px_03
33 bid_sz_03
34 ask_sz_03
35 bid_ct_03
36 ask_ct_03
37 bid_px_04
38 ask_px_04
39 bid_sz_04
40 ask_sz_04
41 bid_ct_04
42 ask_ct_04
43 bid_px_05
44 ask_px_05
45 bid_sz_05
46 ask_sz_05
47 bid_ct_05
48 ask_ct_05
49 bid_px_06
50 ask_px_06
51 bid_sz_06
52 ask_sz_06
53 bid_ct_06
54 ask_ct_06
55 bid_px_07
56 ask_px_07
57 bid_sz_07
58 ask_sz_07
59 bid_ct_07
60 ask_ct_07
61 bid_px_08
62 ask_px_08
63 bid_sz_08
64 ask_sz_08
65 bid_ct_08
66 ask_ct_08
67 bid_px_09
68 ask_px_09
69 bid_sz_09
70 ask_sz_09
71 bid_ct_09
72 ask_ct_09
73 symbol
74 trading_day
75 mid
76 spread
77 spread_bps
78 imbalance_l1
79 depth_l1
8

In [10]:
feat.head(30)

,ts_recv,ts_event,rtype,publisher_id,instrument_id,action,side,depth,price,size,...,imbalance_l10,depth_l10,trade_count,trade_volume,sell_trade_price_min,sell_trade_volume,buy_trade_price_max,buy_trade_volume,rvol,regime
0,2026-06-01 08:00:00+00:00,2026-06-01 08:00:00.486798746+00:00,10.0,2.0,11667.0,C,A,0.0,216.33,91.0,...,-0.082353,170.0,24,1180.0,215.56,367.0,216.20,13.0,NaN,medium
1,2026-06-01 08:00:00.500000+00:00,2026-06-01 08:00:00.999161448+00:00,10.0,2.0,11667.0,C,B,0.0,215.75,5.0,...,-0.625000,416.0,5,264.0,215.89,51.0,NaN,0.0,NaN,medium
2,2026-06-01 08:00:01+00:00,2026-06-01 08:00:01.497742964+00:00,10.0,2.0,11667.0,C,B,1.0,215.62,1.0,...,0.263804,163.0,2,30.0,NaN,0.0,NaN,0.0,NaN,medium
3,2026-06-01 08:00:01.500000+00:00,2026-06-01 08:00:01.968726516+00:00,10.0,2.0,11667.0,C,A,1.0,215.94,1.0,...,0.026316,152.0,11,155.0,215.54,43.0,NaN,0.0,NaN,medium
4,2026-06-01 08:00:02+00:00,2026-06-01 08:00:02.482731617+00:00,10.0,2.0,11667.0,A,B,6.0,215.50,40.0,...,0.084629,579.0,5,289.0,NaN,0.0,215.70,289.0,NaN,medium
5,2026-06-01 08:00:02.500000+00:00,2026-06-01 08:00:02.974312363+00:00,10.0,2.0,11667.0,A,A,1.0,215.75,25.0,...,-0.148305,472.0,41,428.0,215.64,14.0,215.70,414.0,NaN,medium
6,2026-06-01 08:00:03+00:00,2026-06-01 08:00:03.204399101+00:00,10.0,2.0,11667.0,C,B,0.0,215.66,58.0,...,-0.293556,419.0,0,0.0,NaN,0.0,NaN,0.0,NaN,medium
7,2026-06-01 08:00:04+00:00,2026-06-01 08:00:04.485891469+00:00,10.0,2.0,11667.0,C,A,0.0,215.70,3.0,...,-0.409182,501.0,3,7.0,NaN,0.0,215.70,7.0,NaN,medium
8,2026-06-01 08:00:04.500000+00:00,2026-06-01 08:00:04.734772018+00:00,10.0,2.0,11667.0,C,A,0.0,215.70,14.0,...,-0.443478,460.0,10,330.0,NaN,0.0,215.70,330.0,NaN,medium
9,2026-06-01 08:00:05+00:00,2026-06-01 08:00:05.486084217+00:00,10.0,2.0,11667.0,T,N,0.0,215.68,110.0,...,-0.371007,407.0,8,364.0,NaN,0.0,215.70,253.0,NaN,medium


In [9]:
feat["spread"].describe()

count    1.409986e+06
mean     4.331252e-02
std      8.785505e-02
min      1.000000e-02
25%      2.000000e-02
50%      2.000000e-02
75%      4.000000e-02
max      4.969000e+01
Name: spread, dtype: float64